In [1]:

import os

print("Current folder:", os.getcwd())
print("Files in this folder:", os.listdir())


Current folder: C:\Users\HEMA\Flood-Early-Warning\notebooks
Files in this folder: ['.ipynb_checkpoints', 'inspect_gpm_data.ipynb']


In [2]:

import os

project_path = r"C:\Users\HEMA\Flood-Early-Warning"

for folder in ["datasets", "datasets/raw", "datasets/processed"]:
    path = os.path.join(project_path, folder)
    print("\nFolder:", folder)
    if os.path.exists(path):
        print(os.listdir(path))
    else:
        print("Folder not found")



Folder: datasets
['processed', 'raw']

Folder: datasets/raw
['3B-HHR.MS.MRG.3IMERG.20200102-S233000-E235959.1410.V07B.HDF5']

Folder: datasets/processed
[]


In [3]:

import h5py
import os

file_path = r"C:\Users\HEMA\Flood-Early-Warning\datasets\raw\3B-HHR.MS.MRG.3IMERG.20200102-S233000-E235959.1410.V07B.HDF5"

with h5py.File(file_path, "r") as f:
    print("Top-level groups and datasets:")
    f.visititems(lambda name, obj: print(name, "|", type(obj).__name__))


Top-level groups and datasets:
Grid | Group
Grid/Intermediate | Group
Grid/Intermediate/IRinfluence | Dataset
Grid/Intermediate/IRprecipitation | Dataset
Grid/Intermediate/MWobservationTime | Dataset
Grid/Intermediate/MWprecipSource | Dataset
Grid/Intermediate/MWprecipitation | Dataset
Grid/Intermediate/precipitationUncal | Dataset
Grid/lat | Dataset
Grid/lat_bnds | Dataset
Grid/latv | Dataset
Grid/lon | Dataset
Grid/lon_bnds | Dataset
Grid/lonv | Dataset
Grid/nv | Dataset
Grid/precipitation | Dataset
Grid/precipitationQualityIndex | Dataset
Grid/probabilityLiquidPrecipitation | Dataset
Grid/randomError | Dataset
Grid/time | Dataset
Grid/time_bnds | Dataset


In [4]:

import h5py
import numpy as np
import os

file_path = r"C:\Users\HEMA\Flood-Early-Warning\datasets\raw\3B-HHR.MS.MRG.3IMERG.20200102-S233000-E235959.1410.V07B.HDF5"

with h5py.File(file_path, "r") as f:
    grid = f["Grid"]

    for name in ["precipitation", "lat", "lon", "time"]:
        data = grid[name][:]
        print(f"\n{name}")
        print("Shape:", data.shape)
        print("Data type:", data.dtype)
        print("First values:", data.flatten()[:10])

    lat = grid["lat"][:]
    lon = grid["lon"][:]
    rain = grid["precipitation"][:]

    print("\nLatitude range:", np.min(lat), "to", np.max(lat))
    print("Longitude range:", np.min(lon), "to", np.max(lon))
    print("Rainfall minimum:", np.nanmin(rain))
    print("Rainfall maximum:", np.nanmax(rain))
    print("Missing/invalid values:", np.sum(~np.isfinite(rain)))



precipitation
Shape: (1, 3600, 1800)
Data type: float32
First values: [-9999.9 -9999.9 -9999.9 -9999.9     0.      0.      0.      0.      0.
     0. ]

lat
Shape: (1800,)
Data type: float32
First values: [-89.95     -89.85     -89.75     -89.65     -89.549995 -89.45
 -89.35     -89.25     -89.15     -89.049995]

lon
Shape: (3600,)
Data type: float32
First values: [-179.95    -179.85    -179.75    -179.65    -179.55    -179.45
 -179.34999 -179.25    -179.15    -179.05   ]

time
Shape: (1,)
Data type: int32
First values: [1262043000]

Latitude range: -89.95 to 89.95
Longitude range: -179.95 to 179.95
Rainfall minimum: -9999.9
Rainfall maximum: 44.92
Missing/invalid values: 0


In [5]:

import h5py
import numpy as np
from datetime import datetime, timezone

file_path = r"C:\Users\HEMA\Flood-Early-Warning\datasets\raw\3B-HHR.MS.MRG.3IMERG.20200102-S233000-E235959.1410.V07B.HDF5"

with h5py.File(file_path, "r") as f:
    grid = f["Grid"]

    lat = grid["lat"][:]
    lon = grid["lon"][:]
    rain = grid["precipitation"][0, :, :]

    # Approximate bounding box around the Koshi Basin
    lat_mask = (lat >= 25.0) & (lat <= 29.5)
    lon_mask = (lon >= 85.0) & (lon <= 89.5)

    region = rain[np.ix_(lon_mask, lat_mask)]

    # Exclude the likely fill value
    valid = np.isfinite(region) & (region != -9999.9)

    print("Latitude cells:", np.sum(lat_mask))
    print("Longitude cells:", np.sum(lon_mask))
    print("Regional grid shape:", region.shape)
    print("Valid grid cells:", np.sum(valid))

    if np.any(valid):
        print("Minimum stored value:", np.min(region[valid]))
        print("Maximum stored value:", np.max(region[valid]))
    else:
        print("No valid rainfall cells found in this region.")

    timestamp = int(grid["time"][0])
    print("Timestamp (UTC):",
          datetime.fromtimestamp(timestamp, timezone.utc))


Latitude cells: 45
Longitude cells: 45
Regional grid shape: (45, 45)
Valid grid cells: 2025
Minimum stored value: 0.0
Maximum stored value: 2.26
Timestamp (UTC): 2009-12-28 23:30:00+00:00


In [6]:

import h5py

file_path = r"C:\Users\HEMA\Flood-Early-Warning\datasets\raw\3B-HHR.MS.MRG.3IMERG.20200102-S233000-E235959.1410.V07B.HDF5"

with h5py.File(file_path, "r") as f:
    print("FILE ATTRIBUTES")
    for key, value in f.attrs.items():
        print(key, ":", value)

    print("\nGRID ATTRIBUTES")
    for key, value in f["Grid"].attrs.items():
        print(key, ":", value)

    print("\nIMPORTANT DATASET ATTRIBUTES")
    for name in ["precipitation", "lat", "lon", "time"]:
        dataset = f["Grid"][name]
        print(f"\n{name}:")
        for key, value in dataset.attrs.items():
            print(key, ":", value)


FILE ATTRIBUTES
FileHeader : b'DOI=10.5067/GPM/IMERG/3B-HH/07;\nDOIauthority=http://dx.doi.org/;\nDOIshortName=3IMERGHH;\nAlgorithmID=3IMERGHH;\nAlgorithmVersion=3IMERGH_7.0;\nFileName=3B-HHR.MS.MRG.3IMERG.20200102-S233000-E235959.1410.V07B.HDF5;\nSatelliteName=MULTI;\nInstrumentName=MERGED;\nGenerationDateTime=2023-12-30T18:40:56.000Z;\nStartGranuleDateTime=2020-01-02T23:30:00.000Z;\nStopGranuleDateTime=2020-01-02T23:59:59.999Z;\nGranuleNumber=;\nNumberOfSwaths=0;\nNumberOfGrids=1;\nGranuleStart=;\nTimeInterval=HALF_HOUR;\nProcessingSystem=PPS;\nProductVersion=V07B;\nEmptyGranule=NOT_EMPTY;\nMissingData=;\n'
FileInfo : b'DataFormatVersion=7e;\nTKCodeBuildVersion=0;\nMetadataVersion=7e;\nFormatPackage=HDF5-1.10.5;\nBlueprintFilename=GPM.V7.3IMERGHH.blueprint.xml;\nBlueprintVersion=BV_69;\nTKIOVersion=3.100;\nMetadataStyle=PVL;\nEndianType=LITTLE_ENDIAN;\n'

GRID ATTRIBUTES
GridHeader : b'BinMethod=ARITHMETIC_MEAN;\nRegistration=CENTER;\nLatitudeResolution=0.1;\nLongitudeResolution=0.1;

In [7]:

import h5py
from datetime import datetime, timedelta, timezone

file_path = r"C:\Users\HEMA\Flood-Early-Warning\datasets\raw\3B-HHR.MS.MRG.3IMERG.20200102-S233000-E235959.1410.V07B.HDF5"

with h5py.File(file_path, "r") as f:
    grid = f["Grid"]

    time_value = int(grid["time"][0])
    time_units = grid["time"].attrs["Units"].decode()

    origin_text = time_units.replace("seconds since ", "")
    origin = datetime.strptime(
        origin_text, "%Y-%m-%d %H:%M:%S UTC"
    ).replace(tzinfo=timezone.utc)

    observation_time = origin + timedelta(seconds=time_value)

    print("Time units:", time_units)
    print("Raw time value:", time_value)
    print("Decoded observation time:", observation_time)

    print("\nFilename start time:")
    print("2020-01-02 23:30:00 UTC")


Time units: seconds since 1980-01-06 00:00:00 UTC
Raw time value: 1262043000
Decoded observation time: 2020-01-02 23:30:00+00:00

Filename start time:
2020-01-02 23:30:00 UTC


In [8]:

import h5py
import numpy as np
import pandas as pd
import os

project_path = r"C:\Users\HEMA\Flood-Early-Warning"
file_path = os.path.join(
    project_path, "datasets", "raw",
    "3B-HHR.MS.MRG.3IMERG.20200102-S233000-E235959.1410.V07B.HDF5"
)
output_path = os.path.join(
    project_path, "datasets", "processed",
    "koshi_rainfall_sample.csv"
)

with h5py.File(file_path, "r") as f:
    grid = f["Grid"]

    lat = grid["lat"][:]
    lon = grid["lon"][:]
    rain = grid["precipitation"][0, :, :]

    # Approximate regional bounding box
    lat_mask = (lat >= 25.0) & (lat <= 29.5)
    lon_mask = (lon >= 85.0) & (lon <= 89.5)

    region = rain[np.ix_(lon_mask, lat_mask)]
    lon_region = lon[lon_mask]
    lat_region = lat[lat_mask]

    lon_grid, lat_grid = np.meshgrid(
        lon_region, lat_region, indexing="ij"
    )

    # Exclude the documented fill value
    valid = np.isfinite(region) & (region != -9999.9)

    df = pd.DataFrame({
        "longitude": lon_grid[valid],
        "latitude": lat_grid[valid],
        "rainfall_mm_per_hour": region[valid]
    })

    df["observation_time_utc"] = "2020-01-02 23:30:00"

    df.to_csv(output_path, index=False)

print("CSV saved successfully!")
print("Output:", output_path)
print("Valid grid cells saved:", len(df))
print("\nFirst five rows:")
print(df.head())


CSV saved successfully!
Output: C:\Users\HEMA\Flood-Early-Warning\datasets\processed\koshi_rainfall_sample.csv
Valid grid cells saved: 2025

First five rows:
   longitude   latitude  rainfall_mm_per_hour observation_time_utc
0  85.049995  25.049999                   0.0  2020-01-02 23:30:00
1  85.049995  25.150000                   0.0  2020-01-02 23:30:00
2  85.049995  25.250000                   0.0  2020-01-02 23:30:00
3  85.049995  25.350000                   0.0  2020-01-02 23:30:00
4  85.049995  25.449999                   0.0  2020-01-02 23:30:00


In [9]:

import pandas as pd
import os

csv_path = r"C:\Users\HEMA\Flood-Early-Warning\datasets\processed\koshi_rainfall_sample.csv"

df = pd.read_csv(csv_path)

print("Number of rows:", len(df))
print("Column names:", df.columns.tolist())
print("\nMissing values:")
print(df.isnull().sum())
print("\nRainfall summary:")
print(df["rainfall_mm_per_hour"].describe())
print("\nUnique observation times:")
print(df["observation_time_utc"].unique())


Number of rows: 2025
Column names: ['longitude', 'latitude', 'rainfall_mm_per_hour', 'observation_time_utc']

Missing values:
longitude               0
latitude                0
rainfall_mm_per_hour    0
observation_time_utc    0
dtype: int64

Rainfall summary:
count    2025.000000
mean        0.137886
std         0.392708
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         2.260000
Name: rainfall_mm_per_hour, dtype: float64

Unique observation times:
<StringArray>
['2020-01-02 23:30:00']
Length: 1, dtype: str


In [10]:

import os

project_path = r"C:\Users\HEMA\Flood-Early-Warning"

folders = [
    os.path.join(project_path, "datasets", "raw", "gpm_historical"),
    os.path.join(project_path, "datasets", "processed")
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print("Folder ready:", folder)


Folder ready: C:\Users\HEMA\Flood-Early-Warning\datasets\raw\gpm_historical
Folder ready: C:\Users\HEMA\Flood-Early-Warning\datasets\processed
